# Savant 01, real data

This notebook runs Savant on the real tables in the private savant-real dataset. Each table is cleaned with a report of every change, profiled, scanned by the guarded engine, and then answered by the base model in guided mode. Nothing here is graded against a known truth, so the output is meant to be read and judged by a person.

In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import glob
import json
import subprocess
import sys
import time
from pathlib import Path

build = sorted(glob.glob("/kaggle/input/**/BUILD", recursive=True))
assert build and Path(build[0]).read_text().strip() == "EXPECTED_BUILD", "stale bench dataset, stop"
wheel = sorted(glob.glob("/kaggle/input/**/savant-*.whl", recursive=True))[-1]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", wheel], check=True)
FILES = sorted(
    p
    for p in glob.glob("/kaggle/input/**/savant-real/**/*", recursive=True)
    if p.lower().endswith((".csv", ".tsv", ".xlsx", ".xls", ".parquet", ".json", ".jsonl"))
)
print(len(FILES), "files", *FILES, sep="\n")

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
QUESTION = (
    "What are the most important relationships between columns in this data? "
    "Report only what the data supports. End your answer with one line per relationship "
    "in the form PAIR: column_a | column_b, or the single line PAIR: none."
)
MAX_NEW_TOKENS = 768
KEEP_MESSAGES = 6

In [ ]:
from savant.agent import findings_text
from savant.clean import clean
from savant.engine import Settings, analyze
from savant.ingest import load

tables = {}
summary = {}
for path in FILES:
    name = Path(path).stem
    t0 = time.time()
    df, report = clean(load(path, raw=True))
    res = analyze(df, Settings(stability_runs=100))
    tables[name] = df
    summary[name] = {
        "rows": df.height,
        "columns": df.width,
        "seconds": round(time.time() - t0, 1),
        "cleaning": report.to_text(),
        "entity": res.card.entity_column,
        "time": res.card.time_column,
        "kinds": {c.name: c.kind for c in res.card.columns},
        "tests": len(res.ledger),
        "dropped": res.dropped,
        "findings": findings_text(res),
        "confirmed": len(res.reported),
    }
    print(f"===== {name}  {df.height} rows, {df.width} columns, {summary[name]['seconds']}s")
    print(report.to_text())
    print(res.card.to_text())
    print(f"{len(res.ledger)} tests, dropped {res.dropped}")
    print(findings_text(res), end="\n\n")

In [ ]:
import torch

from savant.agent import ask, parse_pairs
from savant.llm import Transformers

client = Transformers(MODEL_ID, max_new_tokens=MAX_NEW_TOKENS)
for name, df in tables.items():
    t0 = time.time()
    try:
        r = ask(df, QUESTION, client, keep_messages=KEEP_MESSAGES, show_findings=True)
        answer, grounded, rounds = r.rendered, r.grounded, r.rounds
    except torch.cuda.OutOfMemoryError as e:
        answer, grounded, rounds = f"out of memory: {e}"[:300], False, None
    torch.cuda.empty_cache()
    summary[name]["model"] = {
        "grounded": grounded,
        "rounds": rounds,
        "seconds": round(time.time() - t0, 1),
        "pairs": sorted(" | ".join(sorted(p)) for p in parse_pairs(answer or "", df.columns)),
        "answer": answer,
    }
    print(f"===== {name}", "grounded" if grounded else "NOT grounded", rounds, "rounds")
    print(answer, end="\n\n")
Path("/kaggle/working/real.json").write_text(json.dumps(summary, indent=1, default=str))